# CSE465 ColorBench — Color Illusion Improvement Pipeline

**Architecture:** Agentic Context Engineering (ACE Framework — ICLR 2026, Stanford/SambaNova/Berkeley)
- **Roles:** Generator $\rightarrow$ Reflector $\rightarrow$ Curator $\rightarrow$ Persistent Playbook $\rightarrow$ Qwen2.5-VL-7B Solver.
- **Target Task:** Color Illusion on ColorBench (UMD, 2025).
- **Validation:** A candidate rule is committed only when its exact curated update adds at least one correct answer on a fixed same-subtype probe set.
- **Fair Evaluation:** Baseline and ACE are evaluated on the **exact same held-out evaluation instances** using fixed-seed splits.
- **Hardware Target:** Google Colab Free-Tier (Tesla T4 GPU, 15GB VRAM) in 4-bit NF4 precision.


## 1. Setup Environment, Google Drive & Run Tag


In [ ]:
# ==============================================================================
# 1. Configuration: RUN_TAG & Google Drive Setup
# ==============================================================================
RUN_TAG = "run-12"  # Change this tag for each experiment run (e.g. run-12, run-13)
OUTPUT_DIR = f"results/{RUN_TAG}"  # Use a new RUN_TAG for each independent experiment.

import os
import shutil
import torch

# 1. Check GPU
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[GPU Ready] {gpu_name} ({vram:.1f} GB VRAM)")
else:
    print("WARNING: No GPU detected. Change Runtime type to GPU (T4) in Colab menu.")

# 2. Mount Google Drive (if on Google Colab)
GDRIVE_BASE_DIR = "/content/drive/MyDrive/CSE465_ColorBench_Results"
GDRIVE_RUN_DIR = os.path.join(GDRIVE_BASE_DIR, RUN_TAG)

try:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs(GDRIVE_RUN_DIR, exist_ok=True)
    print(f"[Google Drive] Mounted successfully! Output will sync to: {GDRIVE_RUN_DIR}")
except Exception as e:
    print(f"[Google Drive Note] Drive mount skipped or not on Colab: {e}")
    GDRIVE_RUN_DIR = os.path.abspath(f"./drive_backup/{RUN_TAG}")
    os.makedirs(GDRIVE_RUN_DIR, exist_ok=True)

# Helper function to sync all results, playbooks, and terminal logs to Google Drive
def sync_results_to_gdrive(run_tag=RUN_TAG, gdrive_dir=GDRIVE_RUN_DIR):
    """Copy all results, playbooks, and cell output logs to Google Drive."""
    local_results_dir = os.path.abspath("./results")
    if not os.path.exists(local_results_dir):
        print(f"[Sync] No local results found at {local_results_dir}")
        return
    
    os.makedirs(gdrive_dir, exist_ok=True)
    copied_files = []
    for root, dirs, files in os.walk(local_results_dir):
        for f in files:
            src_path = os.path.join(root, f)
            rel_path = os.path.relpath(src_path, local_results_dir)
            dst_path = os.path.join(gdrive_dir, rel_path)
            os.makedirs(os.path.dirname(dst_path), exist_ok=True)
            shutil.copy2(src_path, dst_path)
            copied_files.append(rel_path)
            
    print(f"\n[Google Drive Sync] Successfully copied {len(copied_files)} file(s) to:\n  -> {gdrive_dir}")
    for cf in sorted(copied_files):
        print(f"     ✓ {cf}")


## 2. Install Dependencies


In [ ]:
!pip install -q transformers bitsandbytes accelerate qwen-vl-utils datasets "pillow<11.0.0"


## 3. Clone / Pull Repository (`tanzim-ace` Branch)


In [ ]:
import os

# Update with your repository URL
REPO_URL = "https://github.com/tanzim12911/cse465-project.git"

%cd /content
if not os.path.exists("cse465-project"):
    !git clone -b tanzim-ace {REPO_URL}
else:
    %cd cse465-project
    !git checkout tanzim-ace
    !git pull origin tanzim-ace

%cd /content/cse465-project
!git status


## 4. Run Unit Tests (Verify ACE Components)


In [ ]:
!python -m pytest tests/


## 5. Select Model Configuration

Choose a model for your experiments on Colab T4 GPU:
- `"7b"` → `Qwen/Qwen2.5-VL-7B-Instruct` **(recommended)** — 4-bit NF4, ~7 GB VRAM peak. Better reasoning quality for the Generator/Reflector meta-cognitive tasks in ACE.
- `"3b"` → `Qwen/Qwen2.5-VL-3B-Instruct` — faster but weaker instruction following; higher JSON parse failure rate hurts playbook quality. Use only if hitting OOM on 7B.

**Memory note:** 7B NF4 uses ~7 GB peak on T4 (15 GB). The `min_pixels`/`max_pixels` caps in `config.py` prevent vision token OOM.


In [ ]:
# Select model: "7b" or "3b"
MODEL_ID = "7b"
print(f"Selected Model: {MODEL_ID}")


## 6. Primary Experiment: Color Illusion (Baseline vs. Adaptive Playbooks)

Runs both Baseline and ACE on the exact same held-out split (seed=42):
- **Phase 1 (Adaptation + fixed subtype probes):** 60 samples.
- **Phase 2 (Held-Out Evaluation):** 30 unseen samples, evaluated against the same baseline split.

> **Note:** The pipeline reserves a same-subtype fraction of adaptation examples as probes; only rules with a strict probe gain are committed.


In [ ]:
# Run Color Illusion (baseline + strict subtype-validated adaptive playbooks).
# Output is displayed live and saved under the unique RUN_TAG directory.
!mkdir -p {OUTPUT_DIR}/seed_42
!python run_pipeline.py --model_id {MODEL_ID} --mode both --num_adaptation 60 --num_eval 30 --seed 42 --output_dir {OUTPUT_DIR}/seed_42 2>&1 | tee {OUTPUT_DIR}/seed_42/pipeline.log

# Sync results and pipeline log to Google Drive
sync_results_to_gdrive()


## 7. Repeat Experiment on Another Fixed Seed

The pipeline is Color-Illusion-only and always uses subtype playbooks (uniformity, comparison, and ranking) with strict validation.

Use a different seed and a new run tag to measure variation; do not overwrite or resume the previous run.

> Compare the held-out baseline and adaptive scores across several seeds before concluding that a change helps.


In [ ]:
# Repeat only after changing RUN_TAG above (for example, "run-13").
# This cell uses seed 7 and writes to the run-specific output directory.
!mkdir -p {OUTPUT_DIR}/seed_7
!python run_pipeline.py --model_id {MODEL_ID} --mode both --num_adaptation 60 --num_eval 30 --seed 7 --output_dir {OUTPUT_DIR}/seed_7 2>&1 | tee {OUTPUT_DIR}/seed_7/pipeline.log

# Sync results and the second pipeline log to Google Drive
sync_results_to_gdrive()


## 8. Comparative Evaluation Report

Summarizes held-out baseline accuracy vs. held-out ACE accuracy and delta improvements.


In [ ]:
# Run comparative evaluation report
# Output is displayed live and saved to results/output_cell8_eval_report.log
!python eval_results.py --dir ./results 2>&1 | tee results/output_cell8_eval_report.log

# Final sync of all results, playbooks, and all cell output logs to Google Drive
sync_results_to_gdrive()


## 9. Inspect What ACE Learned (Evolved Playbooks)


In [ ]:
import glob
from IPython.display import display, Markdown

# Search recursively so both flat (results/playbook_*.md) and
# run-subfolder (results/run-N/playbook_*.md) layouts are covered.
playbook_md_files = glob.glob("results/**/playbook_*.md", recursive=True)
playbook_md_files += glob.glob("results/playbook_*.md")
playbook_md_files = list(dict.fromkeys(playbook_md_files))  # deduplicate

if not playbook_md_files:
    print("No playbook markdown files found under results/.")
else:
    for md_path in playbook_md_files:
        print(f"\n{'='*70}\nInspecting Playbook: {md_path}\n{'='*70}")
        with open(md_path, "r", encoding="utf-8") as f:
            display(Markdown(f.read()))
